# Decoding a Language Model

A large language model (LLM) doesn't "answer" directly: for a given text, it computes a probability for every *token* (piece of a word) that could come next. How the next token is chosen, the **decoding**, changes the result a lot.

We use a small open model here, [Qwen2.5](https://huggingface.co/Qwen/Qwen2.5-1.5B) (Apache 2.0 license). A GPU speeds up the demonstration (`Runtime > Change runtime type`), but it also works on CPU.

In [ ]:
!pip install -q "transformers>=4.56,<5" accelerate

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "Qwen/Qwen2.5-1.5B"
CHAT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype="auto").to(device)
print(device)

## Tokens, Not Words

The model doesn't see words but *tokens*: pieces of words from a fixed vocabulary.

In [ ]:
SENTENCE = "Artificial intelligence is transforming companies."

print(tokenizer.tokenize(SENTENCE))

## The Probability of the Next Token

Given the beginning of a sentence, the model gives a probability to each of the ~150,000 tokens of its vocabulary. Here are the 10 most likely ones:

In [ ]:
CAPITAL_PROMPT = "The Eiffel Tower is located in"

def next_token_table(prompt: str, k: int = 10) -> pd.DataFrame:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    probabilities = torch.softmax(logits.float(), dim=-1)
    top = torch.topk(probabilities, k)
    return pd.DataFrame({
        "token": [repr(tokenizer.decode(i)) for i in top.indices],
        "probability": top.values.cpu().numpy().round(3),
    })


next_token_table(CAPITAL_PROMPT)

The model learned these probabilities by reading huge amounts of text: it has no base of facts, only statistical regularities.

## Greedy Decoding

The simplest strategy: take the most likely token at each step, append it to the text, and repeat.

In [ ]:
STORY_PROMPT = "The best way to learn a foreign language is"

def greedy(prompt: str, n_tokens: int = 40) -> str:
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]
        next_id = logits.argmax()
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return tokenizer.decode(ids[0], skip_special_tokens=True)


print(greedy(STORY_PROMPT))

Greedy decoding is deterministic: the same beginning always gives the same continuation, often bland, and it sometimes loops.

## Sampling and Temperature

Rather than taking the most likely token, we can **draw** the next token according to the model's probabilities. The **temperature** sets the boldness: low, we stick to the most likely tokens; high, we allow unlikely choices.

In [ ]:
def generate(prompt: str, **kwargs) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id,
                                **kwargs)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
for temperature in [0.2, 0.8, 1.5]:
    set_seed(0)
    print(f"--- temperature = {temperature}")
    print(generate(STORY_PROMPT, do_sample=True, temperature=temperature, top_k=0))

At low temperature the text is safe; at high temperature it becomes creative, then incoherent.

## Top-p: Cutting Unlikely Choices

*Top-p* (or *nucleus sampling*) only draws among the most likely tokens whose cumulative probability reaches `p`: we keep variety without derailing. It is the default setting of most assistants. Three draws, three different texts:

In [ ]:
for seed in range(3):
    set_seed(seed)
    print(f"--- seed = {seed}")
    print(generate(STORY_PROMPT, do_sample=True, temperature=0.8, top_p=0.9))

## Base Model or "Assistant" Model?

A base model only learned to **continue** text. Let's ask it a question:

In [ ]:
QUESTION = "What are three benefits of remote work for a company?"

set_seed(0)
print(generate(QUESTION, do_sample=True, temperature=0.7, top_p=0.9))

It continues the text: sometimes it answers, sometimes it moves on to other questions or drifts, since nothing pushes it to answer. Assistants (ChatGPT, Claude, Le Chat, …) are base models **fine-tuned** to follow instructions, then aligned with human preferences. Same question to the *Instruct* version of the same model:

In [ ]:
chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL, dtype="auto").to(device)

messages = [{"role": "user", "content": QUESTION}]
inputs = chat_tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                            return_dict=True, return_tensors="pt").to(device)
set_seed(0)
with torch.no_grad():
    output = chat_model.generate(**inputs, max_new_tokens=200, do_sample=True,
                                 temperature=0.7, top_p=0.9)
prompt_length = inputs["input_ids"].shape[1]
print(chat_tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True))

## Takeaways

- An LLM computes probabilities of next tokens; the text depends on how we choose among them
- Temperature and top-p set the trade-off between reliability and creativity
- The same prompt can give different answers: keep it in mind when testing and evaluating
- An assistant is a base model fine-tuned to follow instructions